In [1]:
import os
import pandas as pd
import numpy as np
import pickle
import json
from datetime import datetime, timezone

if os.path.exists('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
    project_path = '/content/drive/MyDrive/mimic-sepsis-ews'
else:
    project_path = os.path.abspath(os.path.join(os.getcwd(), '..'))

processed_path = f'{project_path}/data/processed'
print(f"Environment: {'Colab' if 'drive' in project_path else 'Local VSCode'}")

with open(f'{processed_path}/rf_model.pkl', 'rb') as f:
    rf = pickle.load(f)

X_test = pd.read_csv(f'{processed_path}/X_test.csv')
y_test = pd.read_csv(f'{processed_path}/y_test.csv').squeeze()
y_prob = rf.predict_proba(X_test)[:, 1]

print(f"Model loaded successfully")
print(f"Test set: {X_test.shape}")
print(f"Risk score range: {y_prob.min():.4f} to {y_prob.max():.4f}")
print(f"Mean risk score: {y_prob.mean():.4f}")

Environment: Local VSCode
Model loaded successfully
Test set: (5881, 76)
Risk score range: 0.0358 to 0.9737
Mean risk score: 0.3628


In [2]:
def generate_fhir_risk_assessment(
    patient_id: str,
    encounter_id: str,
    risk_score: float,
    top_features: dict,
    prediction_timestamp: str,
    pre_treatment_certified: bool = True
) -> dict:
    """
    Generates a FHIR R4 RiskAssessment resource for sepsis risk prediction.
    Strictly pre-treatment certified — timestamp confirms no treatment
    markers were used in prediction.
    """

    # Risk category thresholds
    if risk_score >= 0.7:
        risk_category = "high"
        risk_display = "High Sepsis Risk"
        risk_code = "H"
    elif risk_score >= 0.4:
        risk_category = "moderate"
        risk_display = "Moderate Sepsis Risk"
        risk_code = "M"
    else:
        risk_category = "low"
        risk_display = "Low Sepsis Risk"
        risk_code = "L"

    # Build FHIR R4 RiskAssessment resource
    resource = {
        "resourceType": "RiskAssessment",
        "id": f"sepsis-ews-{patient_id}-{encounter_id}",
        "status": "final",
        "method": {
            "coding": [{
                "system": "http://snomed.info/sct",
                "code": "416940007",
                "display": "Pre-treatment machine learning risk assessment"
            }]
        },
        "code": {
            "coding": [{
                "system": "http://snomed.info/sct",
                "code": "11552004",
                "display": "Sepsis (disorder)"
            }],
            "text": "Pre-Treatment Sepsis Early Warning Assessment"
        },
        "subject": {
            "reference": f"Patient/{patient_id}",
            "display": f"Patient {patient_id}"
        },
        "encounter": {
            "reference": f"Encounter/{encounter_id}"
        },
        "occurrenceDateTime": prediction_timestamp,
        "prediction": [{
            "outcome": {
                "coding": [{
                    "system": "http://snomed.info/sct",
                    "code": "11552004",
                    "display": "Sepsis"
                }]
            },
            "probabilityDecimal": round(float(risk_score), 4),
            "qualitativeRisk": {
                "coding": [{
                    "system": "http://terminology.hl7.org/CodeSystem/risk-probability",
                    "code": risk_code,
                    "display": risk_display
                }]
            },
            "rationale": f"Pre-treatment Random Forest classifier (AUROC=0.7766) trained on MIMIC-IV. "
                        f"Top contributing features: {', '.join([f'{k} ({v:+.3f})' for k, v in list(top_features.items())[:3]])}. "
                        f"Pre-treatment certified: {pre_treatment_certified}."
        }],
        "extension": [
            {
                "url": "http://sepsis-ews.mimic.org/pre-treatment-certified",
                "valueBoolean": pre_treatment_certified
            },
            {
                "url": "http://sepsis-ews.mimic.org/model-auroc",
                "valueDecimal": 0.7766
            },
            {
                "url": "http://sepsis-ews.mimic.org/model-version",
                "valueString": "mimic-iv-rf-v1.0"
            },
            {
                "url": "http://sepsis-ews.mimic.org/top-features",
                "valueString": json.dumps(top_features)
            }
        ]
    }

    return resource

print("FHIR generator function defined")
print("Resource type: RiskAssessment (FHIR R4)")
print("Risk categories: Low (<0.4) | Moderate (0.4-0.7) | High (>=0.7)")

FHIR generator function defined
Resource type: RiskAssessment (FHIR R4)
Risk categories: Low (<0.4) | Moderate (0.4-0.7) | High (>=0.7)


In [3]:
import shap

# Compute SHAP values for test set sample
print("Computing SHAP values for FHIR feature rationale...")
explainer = shap.TreeExplainer(rf)
sample = X_test.sample(10, random_state=42)
shap_values = explainer(sample)
shap_sepsis = shap_values[:, :, 1].values

# Generate FHIR resources for 10 sample patients
fhir_bundle = {
    "resourceType": "Bundle",
    "id": "sepsis-ews-predictions",
    "type": "collection",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "entry": []
}

feature_names = X_test.columns.tolist()
timestamp = datetime.now(timezone.utc).isoformat()

for i, (idx, row) in enumerate(sample.iterrows()):
    risk_score = rf.predict_proba(row.values.reshape(1, -1))[0][1]

    # Get top 3 SHAP features for this patient
    patient_shap = shap_sepsis[i]
    top_idx = np.argsort(np.abs(patient_shap))[::-1][:3]
    top_features = {
        feature_names[j]: round(float(patient_shap[j]), 4)
        for j in top_idx
    }

    # Generate FHIR resource
    resource = generate_fhir_risk_assessment(
        patient_id=f"P{idx:06d}",
        encounter_id=f"E{idx:06d}",
        risk_score=risk_score,
        top_features=top_features,
        prediction_timestamp=timestamp
    )

    fhir_bundle["entry"].append({
        "resource": resource
    })

    print(f"Patient {i+1}: risk={risk_score:.4f} | "
          f"category={resource['prediction'][0]['qualitativeRisk']['coding'][0]['display']} | "
          f"top feature={list(top_features.keys())[0]}")

print(f"\nBundle contains {len(fhir_bundle['entry'])} RiskAssessment resources")

Computing SHAP values for FHIR feature rationale...


c:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\skl

Patient 1: risk=0.4951 | category=Moderate Sepsis Risk | top feature=vital_measurement_count
Patient 2: risk=0.1949 | category=Low Sepsis Risk | top feature=vital_measurement_count
Patient 3: risk=0.6953 | category=Moderate Sepsis Risk | top feature=vital_measurement_count
Patient 4: risk=0.1802 | category=Low Sepsis Risk | top feature=vital_measurement_count
Patient 5: risk=0.3415 | category=Low Sepsis Risk | top feature=vital_measurement_count
Patient 6: risk=0.3023 | category=Low Sepsis Risk | top feature=vital_measurement_count
Patient 7: risk=0.3242 | category=Low Sepsis Risk | top feature=vital_measurement_count
Patient 8: risk=0.3272 | category=Low Sepsis Risk | top feature=vital_measurement_count
Patient 9: risk=0.7115 | category=High Sepsis Risk | top feature=vital_measurement_count
Patient 10: risk=0.2731 | category=Low Sepsis Risk | top feature=vital_measurement_count

Bundle contains 10 RiskAssessment resources


c:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
c:\Users\shrey\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\utils\validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(


In [4]:
# Save FHIR bundle to Drive
fhir_path = f'{processed_path}/fhir_bundle_sample.json'
with open(fhir_path, 'w') as f:
    json.dump(fhir_bundle, f, indent=2)

print(f"FHIR bundle saved to: {fhir_path}")
print(f"\n{'='*60}")
print("SAMPLE FHIR RiskAssessment Resource (Patient 5 - Moderate Risk):")
print('='*60)
print(json.dumps(fhir_bundle['entry'][4]['resource'], indent=2))

FHIR bundle saved to: d:\mimic-sepsis-ews/data/processed/fhir_bundle_sample.json

SAMPLE FHIR RiskAssessment Resource (Patient 5 - Moderate Risk):
{
  "resourceType": "RiskAssessment",
  "id": "sepsis-ews-P003504-E003504",
  "status": "final",
  "method": {
    "coding": [
      {
        "system": "http://snomed.info/sct",
        "code": "416940007",
        "display": "Pre-treatment machine learning risk assessment"
      }
    ]
  },
  "code": {
    "coding": [
      {
        "system": "http://snomed.info/sct",
        "code": "11552004",
        "display": "Sepsis (disorder)"
      }
    ],
    "text": "Pre-Treatment Sepsis Early Warning Assessment"
  },
  "subject": {
    "reference": "Patient/P003504",
    "display": "Patient P003504"
  },
  "encounter": {
    "reference": "Encounter/E003504"
  },
  "occurrenceDateTime": "2026-06-27T01:50:33.810163+00:00",
  "prediction": [
    {
      "outcome": {
        "coding": [
          {
            "system": "http://snomed.info/sct",


In [5]:
# Summary statistics of FHIR bundle
risk_scores = [e['resource']['prediction'][0]['probabilityDecimal']
               for e in fhir_bundle['entry']]
categories = [e['resource']['prediction'][0]['qualitativeRisk']['coding'][0]['display']
              for e in fhir_bundle['entry']]

from collections import Counter
cat_counts = Counter(categories)

print("FHIR Bundle Summary")
print("="*50)
print(f"Total RiskAssessment resources: {len(fhir_bundle['entry'])}")
print(f"Bundle timestamp: {fhir_bundle['timestamp']}")
print(f"\nRisk Category Distribution:")
for cat, count in sorted(cat_counts.items()):
    print(f"  {cat}: {count} patients")
print(f"\nRisk Score Statistics:")
print(f"  Min:  {min(risk_scores):.4f}")
print(f"  Max:  {max(risk_scores):.4f}")
print(f"  Mean: {sum(risk_scores)/len(risk_scores):.4f}")
print(f"\nFHIR Standard: R4")
print(f"Pre-treatment certified: True")
print(f"Model version: mimic-iv-rf-v1.0")
print(f"AUROC: 0.7766")
print(f"\nBundle saved: fhir_bundle_sample.json")
print(f"\nNotebook 04 complete.")

FHIR Bundle Summary
Total RiskAssessment resources: 10
Bundle timestamp: 2026-06-27T01:50:33.810042+00:00

Risk Category Distribution:
  High Sepsis Risk: 1 patients
  Low Sepsis Risk: 7 patients
  Moderate Sepsis Risk: 2 patients

Risk Score Statistics:
  Min:  0.1802
  Max:  0.7115
  Mean: 0.3845

FHIR Standard: R4
Pre-treatment certified: True
Model version: mimic-iv-rf-v1.0
AUROC: 0.7766

Bundle saved: fhir_bundle_sample.json

Notebook 04 complete.
